# Groups & Events — Inquiries, Sales & Media Response
**End-to-end G&E analytics notebook**

This notebook consolidates the full G&E analysis built across the project, in order:

1. **Setup & load** - three source files, with the data-quality fixes baked in
2. **Glacier sales scorecard** — EXH vs LBE revenue, by year & month (realized sales)
3. **Inquiry / booking funnel** — volume, booking-type mix
4. **Inquiry → Revenue relation** — annual funnel ratios + structural mix (descriptive, *not* causal)
5. **G&E inquiries response model** — Bayesian (PyMC/NumPyro), media → inquiries
6. **Findings & caveats**

---
### Important framing (read before presenting any number)
- **Revenue source is Glacier**, dated by *event date*. It lags the inquiries/spend that drove it by a variable multi-month booking pipeline.
- We therefore **model media → inquiries** (timing is tight), **not media → revenue** (timing is broken — spend↔revenue lag correlation bounced randomly across 0–26 weeks).
- Inquiry↔revenue is related **descriptively only** (annual ratios, mix comparison). No causal claim.
- **Lead Source is excluded** — 95% of revenue is unattributed in CRM, so it can't drive any revenue analysis.
- **2026 is a partial year** on every series — exclude it from trend statements.

## 1. Setup & Load

Set the three Unity Catalog Volume paths below. Files expected:
- Inquiries (lead-level, has `Created On`, `Booking Type`)
- EXH_ModelData (weekly spend + seasonality)
- Glacier report (event-level revenue, 4 LOB sheets)

**Data-quality fixes embedded here** (all discovered during build):
- Glacier sheets have header on **row 7** (Corporate_Events) / **row 8** (others), with title rows above
- Weekly resampling uses **`W-MON`** anchor to match the spend file's Monday weeks (a Sunday default silently produces zero joined rows)
- Glacier filtered to **realized events** (`Date of Event <= today`); ~1,800 future-dated events (out to 2029) excluded

In [0]:
%pip install pymc arviz numpyro jax jaxlib pytensor scikit-learn openpyxl netcdf4 h5netcdf scipy xarray --quiet
dbutils.library.restartPython()

In [0]:
import pandas as pd, numpy as np
import warnings
warnings.simplefilter("ignore")

# ---- CONFIG: set your Unity Catalog Volume paths ----
INQ_FILE     = f"/Volumes/data_exploration_hub/marketing_hub/model_data/GnE_Data/combined_Inquiries.xlsx"
SPEND_FILE   = f"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/EXH_ModelData.xlsx"
GLACIER_FILE = f"/Volumes/data_exploration_hub/marketing_hub/model_data/GnE_Data/GroupsAndEvent107 - Glacier Report 2023-01-01 to 2026-05-25.xlsx"

TODAY   = pd.Timestamp.now().normalize()   # realized-sales cutoff
MAX_LAG = 12                               # adstock memory (weeks)
RUN_MODEL = True                           # section 5 samples live

print("Config set. Today =", TODAY.date())

In [0]:
# ---- Load inquiries ----
inq = pd.read_excel(INQ_FILE)
inq["Created On"] = pd.to_datetime(inq["Created On"], errors="coerce")
inq = inq.dropna(subset=["Created On"])
inq["Year"] = inq["Created On"].dt.year
print(f"Inquiries: {len(inq):,} rows  ({inq['Created On'].min().date()} -> {inq['Created On'].max().date()})")

In [0]:
# ---- Load Glacier (4 LOB sheets, different header rows) ----
GLACIER_SHEETS = {"Corporate_Events": 7, "Group Sales": 8, "Repeat Business": 8, "Rec Room": 8}
frames = []
for sheet, hdr in GLACIER_SHEETS.items():
    d = pd.read_excel(GLACIER_FILE, sheet_name=sheet, header=hdr).dropna(axis=1, how="all")
    d = d[d["Date of Event"].notna()].copy()        # drop title/total rows
    frames.append(d)
glac = pd.concat(frames, ignore_index=True)
glac["Date of Event"] = pd.to_datetime(glac["Date of Event"], errors="coerce")
glac["Total"] = pd.to_numeric(glac["Total"], errors="coerce")

# EXH / LBE split by venue name (100% coverage, confirmed during build)
is_lbe = glac["Theatre Name"].astype(str).str.contains("Rec Room|TRR|Playdium|PDM", case=False, na=False)
glac["Segment"] = np.where(is_lbe, "LBE", "EXH")

# realized events only
n_future = int((glac["Date of Event"] > TODAY).sum())
glac = glac[glac["Date of Event"] <= TODAY].copy()
glac["Year"]  = glac["Date of Event"].dt.year
glac["Month"] = glac["Date of Event"].dt.month
print(f"Glacier: {len(glac):,} realized events  (dropped {n_future:,} future-dated)")
print(f"Revenue total: ${glac['Total'].sum():,.0f}")

In [0]:
# ---- Load spend + seasonality ----
spend_raw = pd.read_excel(SPEND_FILE)
spend_raw = spend_raw.set_index(pd.to_datetime(spend_raw["week_date"]))
print(f"Spend file: {len(spend_raw)} weeks  ({spend_raw.index.min().date()} -> {spend_raw.index.max().date()})")
ge_cols = [c for c in spend_raw.columns if "G&E" in c]
print("G&E spend columns:", ge_cols)

## 2. Glacier Sales Scorecard — EXH vs LBE

The scorecard metric marketing asked for: **\$ EXH Groups Sales / \$ LBE Groups Sales**, realized.

In [0]:
by_year = glac.pivot_table(index="Year", columns="Segment", values="Total",
                           aggfunc="sum", fill_value=0)
by_year["Total"] = by_year.sum(axis=1)
print("SALES BY YEAR (realized):")
print(by_year.round(0).to_string())

seg = glac.groupby("Segment").agg(Events=("Total","size"),
        Revenue=("Total","sum"), Avg_Event=("Total","mean")).round(0)
print("\nSEGMENT SUMMARY:")
print(seg.to_string())

In [0]:
# Year x Month, per segment (seasonality)
for seg_name in ["EXH","LBE"]:
    ym = glac[glac["Segment"]==seg_name].pivot_table(
        index="Year", columns="Month", values="Total", aggfunc="sum", fill_value=0).round(0)
    print(f"\n{seg_name} — SALES BY YEAR x MONTH:")
    print(ym.to_string())

## 3. Inquiry / Booking Funnel

Top-of-funnel demand: inquiry volume and what it's about.

In [0]:
inq_year = inq.groupby("Year").size().rename("Inquiries")
print("INQUIRIES BY YEAR:")
print(inq_year.to_string())

print("\nINQUIRIES BY BOOKING TYPE (top 12):")
print(inq["Booking Type"].value_counts().head(12).to_string())

## 4. Inquiry → Revenue Relation (DESCRIPTIVE — not causal)

Two honest views. **No causal link** is claimed — Glacier revenue is event-dated and lags
inquiries by a variable booking pipeline, so we stay at the aggregate level where lag washes out.

In [0]:
# --- 4a. Annual funnel ratios ---
funnel = pd.DataFrame({
    "Inquiries": inq.groupby("Year").size(),
    "Events":    glac.groupby("Year").size(),
    "Revenue":   glac.groupby("Year")["Total"].sum(),
}).dropna()
funnel["Events/Inq %"] = (funnel["Events"]/funnel["Inquiries"]*100).round(1)
funnel["Rev/Inquiry"]  = (funnel["Revenue"]/funnel["Inquiries"]).round(0)
funnel["Rev/Event"]    = (funnel["Revenue"]/funnel["Events"]).round(0)
print("ANNUAL FUNNEL (2026 partial — exclude from trend):")
print(funnel.to_string())

In [0]:
# --- 4b. Structural mix: what inquiries are about vs where revenue comes from ---
# Different taxonomies -> compare by THEME, not 1:1.
inq_mix = (inq["Booking Type"].value_counts(normalize=True)*100).round(1).rename("Inquiry %").head(12)
rev_mix = (glac.groupby("Type Of Event")["Total"].sum()/glac["Total"].sum()*100)\
          .round(1).sort_values(ascending=False).rename("Revenue %").head(12)
print("INQUIRY MIX — what people ask about:")
print(inq_mix.to_string())
print("\nREVENUE MIX — where the money comes from:")
print(rev_mix.to_string())
print("\nHeadline: Movie ~36% of inquiries but ~3% of revenue; LBE/Private Screening dominate revenue.")

## 5. G&E Inquiries Response Model (Bayesian)

**Target:** weekly inquiries. **Media:** G&E SEM + G&E Social. **Controls:** seasonality.

Geometric adstock + Hill saturation, half-normal media priors (spend can't reduce inquiries).

**Real fit result (165 wks):** `beta_sem` 46 [89% CI 4.4, 110], `beta_soc` 35 [3.9, 75],
adstock ~0.48 both. Both channels show **credible positive inquiry lift** (CIs clear zero),
but intervals are **wide** — direction solid, magnitude not pinned down.

**Saturation note:** Hill diverged badly on this data (~1,200 divergences) — the k/s params
trade off along a ridge the sampler can't navigate at this sample size. The **tanh fallback**
(toggle below) gave 0 divergences, r_hat 1.00, same substantive result. `USE_TANH=True` is the
recommended default here given the divergence behavior.

In [0]:
import pymc as pm
import pytensor.tensor as pt
import arviz as az

USE_TANH = True   # recommended: tanh saturation (0 divergences). Set False for Hill.

# ---- weekly inquiries ----
wk_inq = inq.set_index("Created On").resample("W-MON").size().rename("inquiries")

# ---- spend + seasonality weekly ----
sem = pd.to_numeric(spend_raw["Consideration_G&E_SEM"],  errors="coerce").fillna(0)
soc = pd.to_numeric(spend_raw["Consideration_G&E_Social"],errors="coerce").fillna(0)
seas = pd.get_dummies(spend_raw["season"], prefix="s").astype(float)
seas["stat"] = (spend_raw["stat_holiday_in_week"]=="Y").astype(float)
seas["lw"]   = (spend_raw["is_long_weekend"]=="Y").astype(float)
seas["d2x"]  = spend_raw["days_to_christmas"].astype(float)

df = pd.concat([wk_inq, sem.rename("sem"), soc.rename("soc"), seas], axis=1).dropna()
print(f"Modeling weeks: {len(df)}  ({df.index.min().date()} -> {df.index.max().date()})")

y     = df["inquiries"].values.astype(float)
sem_n = (df["sem"]/df["sem"].max()).values
soc_n = (df["soc"]/df["soc"].max()).values
seas_cols = [c for c in df.columns if c not in ("inquiries","sem","soc")]
Xs = df[seas_cols].values
Xs = (Xs - Xs.mean(0))/(Xs.std(0)+1e-9)

In [0]:
def adstock_pt(x_np, rate):
    """Vectorized geometric adstock (matches recursive form to ~1e-13)."""
    L = MAX_LAG
    w = rate ** pt.arange(L)
    padded = pt.concatenate([pt.zeros(L-1), pt.as_tensor_variable(x_np)])
    idx = pt.arange(len(x_np))[:, None] + pt.arange(L)[None, :]
    return padded[idx].dot(w[::-1])

if RUN_MODEL:
    with pm.Model() as model:
        a_sem = pm.Beta("adstock_sem", 2, 2); a_soc = pm.Beta("adstock_soc", 2, 2)
        sem_ad = adstock_pt(sem_n, a_sem);     soc_ad = adstock_pt(soc_n, a_soc)

        if USE_TANH:
            sat_sem = pm.HalfNormal("sat_sem", 1.5); sat_soc = pm.HalfNormal("sat_soc", 1.5)
            sem_eff = pt.tanh(sem_ad*sat_sem);        soc_eff = pt.tanh(soc_ad*sat_soc)
        else:  # Hill
            k1=pm.Beta("sat_k_sem",2,2); s1=pm.Gamma("sat_s_sem",3,2)
            k2=pm.Beta("sat_k_soc",2,2); s2=pm.Gamma("sat_s_soc",3,2)
            sem_eff = sem_ad**s1/(k1**s1+sem_ad**s1)
            soc_eff = soc_ad**s2/(k2**s2+soc_ad**s2)

        b_sem  = pm.HalfNormal("beta_sem", 50)
        b_soc  = pm.HalfNormal("beta_soc", 50)
        b_seas = pm.Normal("beta_seas", 0, 10, shape=Xs.shape[1])
        base   = pm.Normal("base", y.mean(), y.std())
        csem = pm.Deterministic('contrib_sem', b_sem*sem_eff)
        csoc = pm.Deterministic('contrib_soc', b_soc*soc_eff)
        mu = base + csem + csoc + pt.dot(Xs, b_seas)
        pm.Normal("y", mu=mu, sigma=pm.HalfNormal("sigma", y.std()), observed=y)

        idata = pm.sample(1500, tune=2000, chains=4, target_accept=0.99,
                          nuts_sampler="numpyro", random_seed=42)
    print("Done sampling.")

In [0]:
if RUN_MODEL:
    summ = az.summary(idata, var_names=["beta_sem","beta_soc","adstock_sem","adstock_soc"])
    print(summ.to_string())
    div = int(idata.sample_stats.diverging.sum())
    print(f"\nDivergences: {div}  (want 0)")
    print("r_hat should be ~1.00. Read beta_sem / beta_soc as DIRECTIONAL (wide CIs).")
    if div > 0:
        print("\n** Divergences > 0: if using Hill, set USE_TANH=True and re-run. **")

## 6. Findings & Caveats

**What the data says (defensible):**
1. **G&E media credibly drives inquiries.** Both SEM and Social show positive lift with 89% CIs clearing zero — stronger than LBE Consideration ever did. Magnitude uncertain (wide CIs, thin data).
2. **Sales growing, LBE faster.** EXH and LBE both up YoY; LBE share rose ~37%→41% (2023→25).
3. **Funnel tightening, yield rising.** Inquiries fell ~32% (2023→25) but revenue rose ~29% — revenue per inquiry nearly doubled ($1,262 → $2,392).
4. **Demand ≠ revenue mix.** Movie is ~36% of inquiries but ~3% of revenue; LBE corporate receptions & private screenings drive the money. *Optimizing media for inquiry volume chases the wrong segment.*

**Caveats (state these when presenting):**
- **No media→revenue model** — booking lag makes it unsound (lag correlation was random across 0–26 wk).
- **Inquiry↔revenue is descriptive**, aggregate-level only. Not a tracked path.
- **2026 partial** — excluded from trends.
- **Glacier vs CRM revenue gap** ($155M vs $54M) — Glacier is itemized & complete; CRM `Actual Revenue` was ~half blank. Reconcile before presenting so the jump isn't a surprise.
- **Lead Source unusable** — 95% of revenue unattributed.

**Parked (data-capture gaps, not analysis problems):**
- Funnel↔revenue *row-level* bridge — no shared ID (2.6% name overlap). Unblocks if CRM Opportunity ID is written onto Glacier sales orders.
- Channel-level media mix — would need spend split into more channels.

## 7. Model-Free Gut Check — does the raw data agree?

Before plotting the decomposition, sanity-check the model's "small media effect" against
raw data with no model. Split weeks into low/mid/high G&E spend tertiles and compare raw
inquiry counts.

**Result:** high-spend weeks average ~22% more inquiries than low-spend weeks — but raw
correlation is weak (r≈0.16). That 22% is *unadjusted* (it includes seasonality: spend and
inquiries both rise in peak periods). The model's smaller incremental figure is what remains
after stripping seasonality/baseline/adstock. The two are consistent — the model isn't
understating media, it's removing confounds. This cell pre-empts the "but high-spend weeks
have more inquiries!" challenge.

In [0]:
# Gut check: raw inquiries by spend tertile (no model)
wk_inq_gc = inq.set_index("Created On").resample("W-MON").size().rename("inq")
ge_spend = (pd.to_numeric(spend_raw["Consideration_G&E_SEM"], errors="coerce").fillna(0)
          + pd.to_numeric(spend_raw["Consideration_G&E_Social"], errors="coerce").fillna(0)
          + pd.to_numeric(spend_raw["Conversion_G&E_Social"], errors="coerce").fillna(0)).rename("spend")
gc = pd.concat([wk_inq_gc, ge_spend], axis=1).dropna()
gc["tier"] = pd.qcut(gc["spend"], 3, labels=["Low","Mid","High"])
summary = gc.groupby("tier")["inq"].mean().round(0)
print("Avg weekly inquiries by spend tertile:"); print(summary.to_string())
lo, hi = summary["Low"], summary["High"]
print(f"\nHigh vs Low: {hi-lo:+.0f} inquiries ({hi/lo-1:+.0%}) — unadjusted, includes seasonality")
print(f"Raw corr(spend, inquiries): r = {gc['spend'].corr(gc['inq']):+.3f}  (weak)")

## 8. Stakeholder Plots (matplotlib, deck-ready)

Seven insight plots. All numbers pull live from the data/model above — nothing hardcoded.
Cineplex-styled, static PNG-friendly.

**Caveats rendered ON the plots** (not buried): media-effect plots show wide credible
intervals; decomposition labels the ~97% non-media share; efficiency bars carry CI error bars.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Cineplex Brand Palette (matches MMM notebook) ─────────────────────────────
CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_LGREY = "#F2F2F2"
CPX_DGREY = "#4A4A4A"
CPX_GREEN = "#2E7D32"
CPX_MGREY = "#8A8A8A"
WHITE     = "#FFFFFF"

# ── Matplotlib Style ──────────────────────────────────────────────────────────
# Aspect ratio presets (matching MMM notebook)
RATIOS = {
    "16:9"  : (16, 9),
    "4:3"   : (12, 9),
    "1:1"   : (9, 9),
    "wide"  : (16, 6),
}
def get_figsize(ratio="16:9"):
    return RATIOS[ratio]

plt.rcParams.update({
    "figure.facecolor": WHITE, "figure.dpi": 150,
    "axes.facecolor": WHITE, "axes.edgecolor": CPX_DGREY,
    "axes.linewidth": 1.2, "axes.grid": False,
    "axes.spines.top": True, "axes.spines.right": True,
    "axes.spines.left": True, "axes.spines.bottom": True,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlecolor": "black", "axes.labelsize": 11,
    "axes.labelweight": "bold", "axes.labelcolor": CPX_DGREY,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "xtick.color": CPX_DGREY, "ytick.color": CPX_DGREY,
    "xtick.major.width": 1.2, "ytick.major.width": 1.2,
    "xtick.labelcolor": CPX_DGREY, "ytick.labelcolor": CPX_DGREY,
    "legend.fontsize": 9, "legend.frameon": False,
    "font.family": "sans-serif", "font.size": 10,
    "font.weight": "bold", "axes.titlelocation": "center",
})

FULL_YRS = [2023, 2024, 2025]   # 2026 partial — excluded from plots

# pull series from objects built earlier in the notebook
sc = glac.pivot_table(index="Year", columns="Segment", values="Total", aggfunc="sum", fill_value=0).loc[FULL_YRS]
fn = pd.DataFrame({"Inq": inq.groupby("Year").size(),
                   "Rev": glac.groupby("Year")["Total"].sum()}).loc[FULL_YRS]
fn["RPI"] = fn["Rev"] / fn["Inq"]
imix = (inq["Booking Type"].value_counts(normalize=True) * 100).head(6)
rmix = (glac.groupby("Type Of Event")["Total"].sum() / glac["Total"].sum() * 100).sort_values(ascending=False).head(6)
print("✓ Brand palette & style configured")

In [0]:
# Plot 1 — EXH vs LBE growth
fig, ax = plt.subplots(figsize=get_figsize("16:9"))
x = np.arange(3)
w = 0.38

b1 = ax.bar(x - w/2, sc["EXH"]/1e6, w, label="EXH (Exhibition)", color=CPX_NAVY)
b2 = ax.bar(x + w/2, sc["LBE"]/1e6, w, label="LBE (Rec Room/Playdium)", color=CPX_GOLD)

# Data labels on bars
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        if h >= 0.1:
            ax.text(bar.get_x() + bar.get_width()/2, h/2,
                    f"${h:.1f}M", ha="center", va="center",
                    fontsize=9, fontweight="bold", color=WHITE)

# Total labels above each year
for i in range(3):
    total = sc["EXH"].iloc[i]/1e6 + sc["LBE"].iloc[i]/1e6
    ax.text(i, max(sc["EXH"].iloc[i], sc["LBE"].iloc[i])/1e6 + 0.8,
            f"Total: ${total:.1f}M", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color=CPX_DGREY)

ax.set_xticks(x)
ax.set_xticklabels(FULL_YRS, fontsize=11)
ax.set_ylabel("Revenue ($M)")
ax.set_xlabel("Year")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.0f}M"))
ax.set_title("G&E Realized Sales by Segment — LBE Growing Faster")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [0]:
# Plot 2+3 — funnel inversion (dual axis)
fig, ax = plt.subplots(figsize=get_figsize("16:9"))
ax2 = ax.twinx()

# Bars for inquiry volume
bars = ax.bar(FULL_YRS, fn["Inq"], width=0.55, color=CPX_MGREY, label="Inquiries", zorder=2)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h/2,
            f"{h:,.0f}", ha="center", va="center",
            fontsize=10, fontweight="bold", color=WHITE)

# Line for revenue per inquiry
line = ax2.plot(FULL_YRS, fn["RPI"], color=CPX_GREEN, marker="o", lw=3,
               ms=10, zorder=3, label="Rev per Inquiry")
for yr, rpi in zip(FULL_YRS, fn["RPI"]):
    ax2.annotate(f"${rpi:,.0f}", (yr, rpi), textcoords="offset points",
                 xytext=(0, 14), ha="center", fontsize=10, fontweight="bold",
                 color=CPX_GREEN)

ax.set_xticks(FULL_YRS)
ax.set_xticklabels(FULL_YRS, fontsize=11)
ax.set_ylabel("Inquiries")
ax.set_xlabel("Year")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e3:.0f}K"))
ax2.set_ylabel("Revenue per Inquiry ($)", color=CPX_GREEN)
ax2.tick_params(axis="y", colors=CPX_GREEN)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_title("Inquiries Falling, but Revenue per Inquiry Nearly Doubled")

# Combined legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
leg_elements = [Patch(facecolor=CPX_MGREY, label="Inquiries"),
                Line2D([0], [0], color=CPX_GREEN, lw=3, marker="o", label="Rev/Inquiry")]
ax.legend(handles=leg_elements, loc="upper center")
plt.tight_layout()
plt.show()

In [0]:
# Plot 4 — demand vs revenue mix
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 9))

# Left panel: Inquiry mix
bars1 = a1.barh(list(imix.index)[::-1], list(imix.values)[::-1], color=CPX_NAVY, height=0.6)
for bar in bars1:
    w = bar.get_width()
    a1.text(w + 0.5, bar.get_y() + bar.get_height()/2,
            f"{w:.1f}%", va="center", fontsize=9, fontweight="bold", color=CPX_DGREY)
a1.set_xlabel("% of Inquiries")
a1.set_title("What People Ask About")
a1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

# Right panel: Revenue mix
bars2 = a2.barh(list(rmix.index)[::-1], list(rmix.values)[::-1], color=CPX_RED, height=0.6)
for bar in bars2:
    w = bar.get_width()
    a2.text(w + 0.5, bar.get_y() + bar.get_height()/2,
            f"{w:.1f}%", va="center", fontsize=9, fontweight="bold", color=CPX_DGREY)
a2.set_xlabel("% of Revenue")
a2.set_title("Where the Money Comes From")
a2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

fig.suptitle("Demand ≠ Revenue: Movie ~36% of Inquiries, ~3% of Revenue",
             fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

In [0]:
# Plot 5 — model forest plot (pulls from idata if model ran)
if RUN_MODEL:
    s = az.summary(idata, var_names=["beta_sem", "beta_soc"])
    rows = [
        ("G&E SEM", float(s.loc["beta_sem", "mean"]),
         float(s.loc["beta_sem", "eti89_lb"] if "eti89_lb" in s else s.loc["beta_sem", "hdi_5.5%"]),
         float(s.loc["beta_sem", "eti89_ub"] if "eti89_ub" in s else s.loc["beta_sem", "hdi_94.5%"])),
        ("G&E Social", float(s.loc["beta_soc", "mean"]),
         float(s.loc["beta_soc"].filter(like="lb").iloc[0] if any("lb" in c for c in s.columns) else s.loc["beta_soc"].iloc[2]),
         float(s.loc["beta_soc"].filter(like="ub").iloc[0] if any("ub" in c for c in s.columns) else s.loc["beta_soc"].iloc[3])),
    ]

    fig, ax = plt.subplots(figsize=get_figsize("wide"))
    colors = [CPX_NAVY, CPX_RED]
    for i, (k, mn, lo, hi) in enumerate(rows):
        ax.plot([lo, hi], [i, i], color=colors[i], lw=4, solid_capstyle="round")
        ax.plot(mn, i, "o", color=colors[i], ms=14, zorder=5)
        # Annotate mean
        ax.annotate(f"{mn:.0f}", (mn, i), textcoords="offset points",
                    xytext=(0, 12), ha="center", fontsize=10, fontweight="bold",
                    color=colors[i])
        # Annotate CI bounds
        ax.text(lo - 2, i, f"{lo:.0f}", ha="right", va="center",
                fontsize=9, color=CPX_DGREY)
        ax.text(hi + 2, i, f"{hi:.0f}", ha="left", va="center",
                fontsize=9, color=CPX_DGREY)

    ax.axvline(0, ls="--", color=CPX_MGREY, lw=1.5, zorder=1)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=12)
    ax.set_xlabel("Effect on Weekly Inquiries (89% Credible Interval)")
    ax.set_title("Both Channels: Credible Positive Lift (CIs Clear Zero)")
    plt.tight_layout()
    plt.show()

In [0]:
# Plots 6 & 7 — decomposition + efficiency (computed from model contributions)
if RUN_MODEL:
    post = idata.posterior
    csem = post["contrib_sem"].sum("contrib_sem_dim_0").mean().item() if "contrib_sem" in post else None
    csoc = post["contrib_soc"].sum("contrib_soc_dim_0").mean().item()
    tot = df["inquiries"].sum()
    sem_pct = csem / tot * 100
    soc_pct = csoc / tot * 100
    base_pct = 100 - sem_pct - soc_pct
    sem_eff = csem / df["sem"].sum() * 1000
    soc_eff = csoc / df["soc"].sum() * 1000

    # ── Plot 6: Decomposition (stacked bar) ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 7))
    bot = 0
    segments = [
        ("Baseline + Seasonality", base_pct, CPX_MGREY),
        ("G&E Social", soc_pct, CPX_RED),
        ("G&E SEM", sem_pct, CPX_NAVY),
    ]
    for k, v, c in segments:
        bar = ax.bar("Inquiry Volume", v, bottom=bot, color=c,
                     width=0.55, label=f"{k} ({v:.1f}%)")
        # Label inside the segment if large enough
        if v >= 3:
            ax.text(0, bot + v/2, f"{v:.1f}%",
                    ha="center", va="center", fontsize=12,
                    fontweight="bold", color=WHITE)
        bot += v

    ax.set_ylabel("% of Inquiries")
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=11)
    ax.set_title(f"Paid Media ~{sem_pct+soc_pct:.0f}% of Inquiry Volume; ~{base_pct:.0f}% Non-Media")
    plt.tight_layout()
    plt.show()

    # ── Plot 7: Efficiency ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=get_figsize("16:9"))
    bars = ax.bar(["G&E Social", "G&E SEM"], [soc_eff, sem_eff],
                  color=[CPX_RED, CPX_NAVY], width=0.5)

    # Data labels on top
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.05,
                f"{h:.1f}", ha="center", va="bottom",
                fontsize=11, fontweight="bold", color=CPX_DGREY)

    ax.set_ylabel("Inquiries per $1K Spent")
    ax.set_title("Media Efficiency (Directional — Wide CIs)")
    ax.set_ylim(0, max(soc_eff, sem_eff) * 1.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}"))
    plt.tight_layout()
    plt.show()

    print(f"Decomposition: SEM {sem_pct:.1f}% | Social {soc_pct:.1f}% | Baseline {base_pct:.1f}%")
    print(f"Efficiency: Social {soc_eff:.1f} vs SEM {sem_eff:.1f} inquiries per $1k")

## 9. Improved Seasonality — G&E-Specific Features

**Hypothesis:** The current model borrows seasonality from the Exhibition MMM file (quarterly seasons,
stat holiday positive, days-to-christmas for tentpoles). G&E demand cycles differently:

| Current Feature | Problem for G&E | Replacement |
|---|---|---|
| Quarterly season dummies | Too coarse — misses Dec corporate spike, Jan dead zone | Monthly Fourier harmonics (2 pairs) |
| Stat holiday (positive) | Corporate events *don't* book on holidays; kids travel | Let sign be learned (Normal prior, not assumed positive) |
| Long weekend (positive) | Same issue — events dip on long weekends | Same — free sign |
| Days to Christmas | Designed for movie tentpoles | Remap as corporate-party ramp (peaks 4–6 wks out) |
| *Missing* | School calendar drives kids' parties | School-in-session flag (Sep–Jun) |
| *Missing* | Corporate budget flush | End-of-quarter flag |

**Test:** Re-run the same Bayesian model with G&E-tuned features. Compare:
1. Model fit (LOO/WAIC)
2. CI width on media betas
3. Decomposition shift

In [0]:
# ── G&E-Specific Seasonality Features ─────────────────────────────────────────
import pandas as pd
import numpy as np

# Start from the same weekly inquiry series and spend
wk_inq_v2 = inq.set_index("Created On").resample("W-MON").size().rename("inquiries")
sem_v2 = pd.to_numeric(spend_raw["Consideration_G&E_SEM"], errors="coerce").fillna(0)
soc_v2 = pd.to_numeric(spend_raw["Consideration_G&E_Social"], errors="coerce").fillna(0)

# Week dates for feature engineering
week_dates = spend_raw["week_date"] if "week_date" in spend_raw.columns else spend_raw.index
week_dates = pd.to_datetime(week_dates)

# ── 1. Monthly Fourier harmonics (2 pairs = captures monthly + bimonthly cycles)
day_of_year = week_dates.dt.dayofyear
fourier = pd.DataFrame(index=spend_raw.index)
for k in [1, 2]:
    fourier[f"sin_{k}"] = np.sin(2 * np.pi * k * day_of_year / 365.25)
    fourier[f"cos_{k}"] = np.cos(2 * np.pi * k * day_of_year / 365.25)

# ── 2. School-in-session (Sep 1 – Jun 30 = 1, Jul–Aug = 0)
month = week_dates.dt.month
school_in_session = ((month >= 9) | (month <= 6)).astype(float)

# ── 3. End-of-quarter flag (last 2 weeks of Mar, Jun, Sep, Dec)
week_of_month = (week_dates.dt.day - 1) // 7 + 1
is_quarter_end_month = month.isin([3, 6, 9, 12])
end_of_quarter = (is_quarter_end_month & (week_dates.dt.day >= 15)).astype(float)

# ── 4. Corporate party ramp — peaks 4-6 weeks before Christmas
#    Gaussian bump centered at ~5 weeks before Dec 25
days_to_xmas = spend_raw["days_to_christmas"].astype(float)
corp_party_ramp = np.exp(-0.5 * ((days_to_xmas - 35) / 10) ** 2)  # peak at 5 wks out, σ=10 days

# ── 5. Stat holiday & long weekend — keep but allow NEGATIVE sign (Normal prior later)
stat_holiday = (spend_raw["stat_holiday_in_week"] == "Y").astype(float)
long_weekend = (spend_raw["is_long_weekend"] == "Y").astype(float)

# ── Assemble new feature matrix
seas_v2 = pd.DataFrame({
    "sin_1": fourier["sin_1"].values,
    "cos_1": fourier["cos_1"].values,
    "sin_2": fourier["sin_2"].values,
    "cos_2": fourier["cos_2"].values,
    "school": school_in_session.values,
    "qtr_end": end_of_quarter.values,
    "corp_party_ramp": corp_party_ramp.values,
    "stat_holiday": stat_holiday.values,
    "long_weekend": long_weekend.values,
}, index=spend_raw.index)

# Build combined dataframe
df_v2 = pd.concat([wk_inq_v2, sem_v2.rename("sem"), soc_v2.rename("soc"), seas_v2], axis=1).dropna()
print(f"V2 modeling weeks: {len(df_v2)}  ({df_v2.index.min().date()} -> {df_v2.index.max().date()})")
print(f"Features ({seas_v2.shape[1]}): {list(seas_v2.columns)}")
print(f"\nOld features (7): [s_fall, s_spring, s_summer, s_winter, stat, lw, d2x]")
print(f"New features (9): {list(seas_v2.columns)}")

# Prepare model arrays
y_v2     = df_v2["inquiries"].values.astype(float)
sem_n_v2 = (df_v2["sem"] / df_v2["sem"].max()).values
soc_n_v2 = (df_v2["soc"] / df_v2["soc"].max()).values
seas_cols_v2 = [c for c in df_v2.columns if c not in ("inquiries", "sem", "soc")]
Xs_v2 = df_v2[seas_cols_v2].values
Xs_v2 = (Xs_v2 - Xs_v2.mean(0)) / (Xs_v2.std(0) + 1e-9)
print(f"\n✓ Feature matrix ready: {Xs_v2.shape}")

In [0]:
# ── V2 Model — same structure, G&E seasonality, Normal priors on calendar ─────
import pymc as pm
import pytensor.tensor as pt
import arviz as az

MAX_LAG_V2 = 8

def adstock_pt_v2(x_np, rate):
    L = MAX_LAG_V2
    w = rate ** pt.arange(L)
    padded = pt.concatenate([pt.zeros(L-1), pt.as_tensor_variable(x_np)])
    idx = pt.arange(len(x_np))[:, None] + pt.arange(L)[None, :]
    return padded[idx].dot(w[::-1])

with pm.Model() as model_v2:
    # Adstock
    a_sem = pm.Beta("adstock_sem", 2, 2)
    a_soc = pm.Beta("adstock_soc", 2, 2)
    sem_ad = adstock_pt_v2(sem_n_v2, a_sem)
    soc_ad = adstock_pt_v2(soc_n_v2, a_soc)

    # Saturation (tanh — same as v1)
    sat_sem = pm.HalfNormal("sat_sem", 1.5)
    sat_soc = pm.HalfNormal("sat_soc", 1.5)
    sem_eff = pt.tanh(sem_ad * sat_sem)
    soc_eff = pt.tanh(soc_ad * sat_soc)

    # Media betas (positive — spend can't reduce inquiries)
    b_sem = pm.HalfNormal("beta_sem", 50)
    b_soc = pm.HalfNormal("beta_soc", 50)

    # Seasonality — NORMAL priors (allow negative effects for holidays)
    b_seas = pm.Normal("beta_seas", 0, 15, shape=Xs_v2.shape[1])

    base = pm.Normal("base", y_v2.mean(), y_v2.std())

    # Contributions (tracked for decomposition)
    csem_v2 = pm.Deterministic("contrib_sem", b_sem * sem_eff)
    csoc_v2 = pm.Deterministic("contrib_soc", b_soc * soc_eff)

    mu = base + csem_v2 + csoc_v2 + pt.dot(Xs_v2, b_seas)
    pm.Normal("y", mu=mu, sigma=pm.HalfNormal("sigma", y_v2.std()), observed=y_v2)

    idata_v2 = pm.sample(1500, tune=2000, chains=4, target_accept=0.99,
                         nuts_sampler="numpyro", random_seed=42)

print("✓ V2 model sampling complete.")

In [0]:
# ── Compare V1 (original) vs V2 (G&E seasonality) ────────────────────────────
print("="*70)
print("  MODEL COMPARISON: V1 (Exhibition seasonality) vs V2 (G&E seasonality)")
print("="*70)

# V2 summary
s_v2 = az.summary(idata_v2, var_names=["beta_sem", "beta_soc", "adstock_sem", "adstock_soc"])
print("\n── V2 (G&E Seasonality) Posteriors ──")
print(s_v2.to_string())

div_v2 = int(idata_v2.sample_stats.diverging.sum())
print(f"\nDivergences: {div_v2}  (want 0)")

# V1 for comparison (already computed)
s_v1 = az.summary(idata, var_names=["beta_sem", "beta_soc", "adstock_sem", "adstock_soc"])
print("\n── V1 (Exhibition Seasonality) Posteriors ──")
print(s_v1.to_string())

# CI width comparison
print("\n── CI Width Comparison (narrower = better identified) ──")
for var in ["beta_sem", "beta_soc"]:
    lb_col = "eti89_lb" if "eti89_lb" in s_v1.columns else "hdi_5.5%"
    ub_col = "eti89_ub" if "eti89_ub" in s_v1.columns else "hdi_94.5%"
    ci_v1 = float(s_v1.loc[var, ub_col]) - float(s_v1.loc[var, lb_col])
    ci_v2 = float(s_v2.loc[var, ub_col]) - float(s_v2.loc[var, lb_col])
    pct_change = (ci_v2 - ci_v1) / ci_v1 * 100
    arrow = "↓ narrower" if pct_change < 0 else "↑ wider"
    print(f"  {var}: V1={ci_v1:.1f}  V2={ci_v2:.1f}  ({pct_change:+.1f}% {arrow})")

# LOO comparison
print("\n── Model Fit (LOO-CV) ──")
try:
    loo_v1 = az.loo(idata)
    loo_v2 = az.loo(idata_v2)
    print(f"  V1 LOO elpd: {loo_v1.elpd_loo:.1f} (se={loo_v1.se:.1f})")
    print(f"  V2 LOO elpd: {loo_v2.elpd_loo:.1f} (se={loo_v2.se:.1f})")
    diff = loo_v2.elpd_loo - loo_v1.elpd_loo
    print(f"  Δ(V2−V1): {diff:+.1f}  {'V2 better' if diff > 0 else 'V1 better'}")
except Exception as e:
    print(f"  LOO comparison failed: {e}")

# Seasonality coefficient interpretation
print("\n── V2 Seasonality Coefficients (what G&E calendar actually does) ──")
b_seas_post = idata_v2.posterior["beta_seas"].mean(dim=["chain", "draw"]).values
for i, col in enumerate(seas_cols_v2):
    sign = "+" if b_seas_post[i] > 0 else ""
    print(f"  {col:18s}: {sign}{b_seas_post[i]:.1f}  {'(drives inquiries UP)' if b_seas_post[i] > 0 else '(drives inquiries DOWN)'}")

In [0]:
# ── Side-by-side decomposition: V1 vs V2 ─────────────────────────────────────
if RUN_MODEL:
    # V2 decomposition
    post_v2 = idata_v2.posterior
    csem2 = post_v2["contrib_sem"].sum("contrib_sem_dim_0").mean().item()
    csoc2 = post_v2["contrib_soc"].sum("contrib_soc_dim_0").mean().item()
    tot2 = df_v2["inquiries"].sum()
    sem_pct2 = csem2 / tot2 * 100
    soc_pct2 = csoc2 / tot2 * 100
    base_pct2 = 100 - sem_pct2 - soc_pct2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=get_figsize("16:9"))

    # V1 (left)
    bot = 0
    for k, v, c in [("Baseline", base_pct, CPX_MGREY), ("Social", soc_pct, CPX_RED), ("SEM", sem_pct, CPX_NAVY)]:
        ax1.bar("V1\n(EXH Seasonality)", v, bottom=bot, color=c, width=0.5)
        if v >= 2:
            ax1.text(0, bot + v/2, f"{v:.1f}%", ha="center", va="center",
                     fontsize=10, fontweight="bold", color=WHITE)
        bot += v
    ax1.set_ylim(0, 105)
    ax1.set_ylabel("% of Inquiries")
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax1.set_title("V1: Original Seasonality")

    # V2 (right)
    bot = 0
    for k, v, c in [("Baseline", base_pct2, CPX_MGREY), ("Social", soc_pct2, CPX_RED), ("SEM", sem_pct2, CPX_NAVY)]:
        ax2.bar("V2\n(G&E Seasonality)", v, bottom=bot, color=c, width=0.5)
        if v >= 2:
            ax2.text(0, bot + v/2, f"{v:.1f}%", ha="center", va="center",
                     fontsize=10, fontweight="bold", color=WHITE)
        bot += v
    ax2.set_ylim(0, 105)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax2.set_title("V2: G&E-Tuned Seasonality")

    # Shared legend
    from matplotlib.patches import Patch
    legend_items = [
        Patch(facecolor=CPX_MGREY, label="Baseline + Seasonality"),
        Patch(facecolor=CPX_RED, label="G&E Social"),
        Patch(facecolor=CPX_NAVY, label="G&E SEM"),
    ]
    fig.legend(handles=legend_items, loc="lower center", ncol=3, fontsize=10)
    fig.suptitle("Decomposition: V1 vs V2 — Does Better Seasonality Change Media Attribution?",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.show()

    print(f"\nV1: SEM {sem_pct:.1f}% | Social {soc_pct:.1f}% | Baseline {base_pct:.1f}%")
    print(f"V2: SEM {sem_pct2:.1f}% | Social {soc_pct2:.1f}% | Baseline {base_pct2:.1f}%")
    delta_media = (sem_pct2 + soc_pct2) - (sem_pct + soc_pct)
    print(f"Δ total media share: {delta_media:+.1f}pp")

## 10. V3 — Reducing Baseline Inflation (Multiplicative Model)

**Problem:** V1/V2 baseline absorbs 95–97% credit. Root causes:

| Cause | Why it inflates baseline | V3 Fix |
|---|---|---|
| Additive model structure | Media adds tiny fixed count (~30 inq/wk) to a large baseline (~460/wk) | **Multiplicative (log-link)** — media lifts are % of total, not fixed additive count |
| tanh saturation | Caps media effect at ~0.9 regardless of spend level | **Hill function** with learnable half-saturation — more flexible, less aggressive cap |
| Short adstock window (8 wks) | Media can’t accumulate effects over longer horizons | **12 weeks** carry-over |
| Diffuse priors + no trend | Intercept absorbs everything | Informative priors + explicit trend (in log space) |
| Missing channel | Conversion_G&E_Social unmodeled | Add as combined social |

**Key insight:** The additive model says “media adds 30 inquiries on top of 460.” The multiplicative model says “media lifts inquiries by X%.” The latter naturally gives media more proportional credit because the lift applies to the whole volume.

In [0]:
# ── V3: Multiplicative model (log-link) + Hill saturation + longer adstock ───
import pandas as pd
import numpy as np

# Add Conversion_G&E_Social as combined social channel
conv_soc_v3 = pd.to_numeric(spend_raw["Conversion_G&E_Social"], errors="coerce").fillna(0)

# Build V3 dataframe
df_v3 = pd.concat([
    wk_inq_v2,
    sem_v2.rename("sem"),
    soc_v2.rename("soc_consid"),
    conv_soc_v3.rename("soc_conv"),
    seas_v2
], axis=1).dropna()

# Combine social channels into total social spend
df_v3["soc_total"] = df_v3["soc_consid"] + df_v3["soc_conv"]

# Add explicit linear time trend (normalized 0→1)
n_weeks = len(df_v3)
df_v3["trend"] = np.linspace(0, 1, n_weeks)

# Prepare model arrays
y_v3 = df_v3["inquiries"].values.astype(float)
log_y_v3 = np.log(y_v3)  # log-link target

# Normalize spend to [0, 1]
sem_n_v3 = (df_v3["sem"] / df_v3["sem"].max()).values
soc_n_v3 = (df_v3["soc_total"] / df_v3["soc_total"].max()).values
trend_v3 = df_v3["trend"].values

seas_cols_v3 = list(seas_v2.columns)
Xs_v3 = df_v3[seas_cols_v3].values
Xs_v3 = (Xs_v3 - Xs_v3.mean(0)) / (Xs_v3.std(0) + 1e-9)

print(f"V3 modeling weeks: {len(df_v3)}")
print(f"Response: log(inquiries) — multiplicative model")
print(f"Saturation: Hill function (learnable half-sat K)")
print(f"Adstock: 12-week geometric")
print(f"Social total spend: ${df_v3['soc_total'].sum():,.0f}  (was ${df_v3['soc_consid'].sum():,.0f} in V1/V2)")
print(f"log(y) range: {log_y_v3.min():.2f} – {log_y_v3.max():.2f}  (mean={log_y_v3.mean():.2f})")
print(f"\n✓ V3 arrays ready")

In [0]:
# ── V3: Multiplicative (log-link) + Hill saturation + 12-week adstock ──────
import pymc as pm
import pytensor.tensor as pt
import arviz as az

MAX_LAG_V3 = 12  # longer carry-over

def adstock_pt_v3(x_np, rate):
    L = MAX_LAG_V3
    w = rate ** pt.arange(L)
    padded = pt.concatenate([pt.zeros(L-1), pt.as_tensor_variable(x_np)])
    idx = pt.arange(len(x_np))[:, None] + pt.arange(L)[None, :]
    return padded[idx].dot(w[::-1])

def hill_pt(x, K, n):
    """Hill saturation: x^n / (K^n + x^n). More flexible than tanh."""
    xn = pt.power(x + 1e-8, n)
    return xn / (pt.power(K, n) + xn)

with pm.Model() as model_v3:
    # Adstock (longer window)
    a_sem = pm.Beta("adstock_sem", 3, 2)  # prior nudges toward higher carry-over
    a_soc = pm.Beta("adstock_soc", 3, 2)
    sem_ad = adstock_pt_v3(sem_n_v3, a_sem)
    soc_ad = adstock_pt_v3(soc_n_v3, a_soc)

    # Hill saturation (learnable half-sat K and shape n)
    K_sem = pm.Beta("K_sem", 2, 2)   # half-saturation point in [0,1]
    K_soc = pm.Beta("K_soc", 2, 2)
    n_sem = pm.Gamma("n_sem", 2, 2)  # shape > 0, prior around 1
    n_soc = pm.Gamma("n_soc", 2, 2)
    sem_eff = hill_pt(sem_ad, K_sem, n_sem)
    soc_eff = hill_pt(soc_ad, K_soc, n_soc)

    # Media betas in LOG SPACE — these are % lifts
    # Prior: ~0.15 in log space = ~16% lift at saturation
    # Informed by gut-check: high-spend weeks ~22% above low
    b_sem = pm.HalfNormal("beta_sem", 0.2)
    b_soc = pm.HalfNormal("beta_soc", 0.2)

    # Trend in log space (structural % decline)
    b_trend = pm.Normal("beta_trend", -0.3, 0.2)  # prior: ~-30% total decline

    # Seasonality in log space
    b_seas = pm.Normal("beta_seas", 0, 0.1, shape=Xs_v3.shape[1])

    # Log-baseline (intercept)
    log_base = pm.Normal("log_base", log_y_v3.mean(), 0.2)

    # Contributions tracked for decomposition
    media_sem = pm.Deterministic("media_sem", b_sem * sem_eff)
    media_soc = pm.Deterministic("media_soc", b_soc * soc_eff)
    log_trend = pm.Deterministic("log_trend", b_trend * pt.as_tensor_variable(trend_v3))

    # Multiplicative: log(y) = log_base + trend + seas + media
    log_mu = log_base + media_sem + media_soc + log_trend + pt.dot(Xs_v3, b_seas)

    # Likelihood in log space (Normal on log(y))
    sigma = pm.HalfNormal("sigma", 0.15)
    pm.Normal("log_y", mu=log_mu, sigma=sigma, observed=log_y_v3)

    idata_v3 = pm.sample(1500, tune=2000, chains=4, target_accept=0.99,
                         nuts_sampler="numpyro", random_seed=42)

print("✓ V3 multiplicative model sampling complete.")

In [0]:
# ── V3 Results — Multiplicative Decomposition (counterfactual) ───────────
print("="*70)
print("  V3 RESULTS: Multiplicative Model (Log-Link + Hill + 12wk Adstock)")
print("="*70)

# Posteriors
s_v3 = az.summary(idata_v3, var_names=["beta_sem", "beta_soc", "beta_trend",
                                        "adstock_sem", "adstock_soc",
                                        "K_sem", "K_soc", "n_sem", "n_soc",
                                        "log_base", "sigma"])
print("\n── V3 Posteriors ──")
print(s_v3.to_string())

div_v3 = int(idata_v3.sample_stats.diverging.sum())
print(f"\nDivergences: {div_v3}  (want 0)")

# ── Counterfactual Decomposition ───────────────────────────────────────
# "What would inquiries be WITHOUT media?" vs actual
post_v3 = idata_v3.posterior

# Mean posterior media contributions (in log space)
media_sem_mean = post_v3["media_sem"].mean(dim=["chain", "draw"]).values  # shape (165,)
media_soc_mean = post_v3["media_soc"].mean(dim=["chain", "draw"]).values
log_trend_mean = post_v3["log_trend"].mean(dim=["chain", "draw"]).values

# Actual inquiries
actual_total = y_v3.sum()

# Predicted WITH media (full model)
log_base_mean = float(post_v3["log_base"].mean())
b_seas_mean = post_v3["beta_seas"].mean(dim=["chain", "draw"]).values
log_pred_full = log_base_mean + media_sem_mean + media_soc_mean + log_trend_mean + Xs_v3 @ b_seas_mean
pred_full = np.exp(log_pred_full).sum()

# Predicted WITHOUT media (counterfactual)
log_pred_no_media = log_base_mean + log_trend_mean + Xs_v3 @ b_seas_mean
pred_no_media = np.exp(log_pred_no_media).sum()

# Media incremental = difference
media_incremental = pred_full - pred_no_media
sem_incremental = (np.exp(log_pred_full) - np.exp(log_pred_full - media_sem_mean)).sum()
soc_incremental = (np.exp(log_pred_full) - np.exp(log_pred_full - media_soc_mean)).sum()

# Shares against ACTUAL total
sem_pct3 = sem_incremental / actual_total * 100
soc_pct3 = soc_incremental / actual_total * 100
media_pct3 = media_incremental / actual_total * 100
base_pct3 = 100 - media_pct3

print("\n── Counterfactual Decomposition ──")
print(f"  G&E SEM contribution:    {sem_pct3:.1f}%  ({sem_incremental:,.0f} incremental inquiries)")
print(f"  G&E Social contribution: {soc_pct3:.1f}%  ({soc_incremental:,.0f} incremental inquiries)")
print(f"  Total media:             {media_pct3:.1f}%  ({media_incremental:,.0f} incremental inquiries)")
print(f"  Non-media (baseline):    {base_pct3:.1f}%")

print("\n── Comparison across all 3 versions ──")
print(f"  V1 media share:  {sem_pct + soc_pct:.1f}%  (additive, EXH seasonality, tanh, diffuse)")
print(f"  V2 media share:  {sem_pct2 + soc_pct2:.1f}%  (additive, G&E seasonality, tanh, diffuse)")
print(f"  V3 media share: {media_pct3:.1f}%  (multiplicative, Hill, 12wk adstock, informed)")

# Interpret beta in log space as % lift
b_sem_mean = float(s_v3.loc["beta_sem", "mean"])
b_soc_mean = float(s_v3.loc["beta_soc", "mean"])
print(f"\n── Beta interpretation (log-space → % lift at saturation) ──")
print(f"  SEM: beta={b_sem_mean:.3f} → at full saturation, SEM lifts inquiries by ~{(np.exp(b_sem_mean)-1)*100:.1f}%")
print(f"  Soc: beta={b_soc_mean:.3f} → at full saturation, Social lifts inquiries by ~{(np.exp(b_soc_mean)-1)*100:.1f}%")

# Trend
b_trend_mean = float(s_v3.loc["beta_trend", "mean"])
print(f"\n── Trend: {b_trend_mean:.3f} in log-space → {(np.exp(b_trend_mean)-1)*100:.1f}% total structural decline")

In [0]:
# ── V1 vs V2 vs V3 decomposition side-by-side ────────────────────────────
if RUN_MODEL:
    from matplotlib.patches import Patch

    fig, axes = plt.subplots(1, 3, figsize=get_figsize("16:9"), sharey=True)

    versions = [
        ("V1\n(Additive/tanh)", [
            ("Non-Media", base_pct, CPX_MGREY),
            ("Social", soc_pct, CPX_RED),
            ("SEM", sem_pct, CPX_NAVY),
        ]),
        ("V2\n(Additive/G&E)", [
            ("Non-Media", base_pct2, CPX_MGREY),
            ("Social", soc_pct2, CPX_RED),
            ("SEM", sem_pct2, CPX_NAVY),
        ]),
        ("V3\n(Multiplicative/Hill)", [
            ("Non-Media", base_pct3, CPX_MGREY),
            ("Social", soc_pct3, CPX_RED),
            ("SEM", sem_pct3, CPX_NAVY),
        ]),
    ]

    for ax, (label, segments) in zip(axes, versions):
        bot = 0
        for name, val, color in segments:
            ax.bar(label, val, bottom=bot, color=color, width=0.5)
            if val >= 2:
                ax.text(0, bot + val/2, f"{val:.1f}%", ha="center", va="center",
                        fontsize=10, fontweight="bold", color=WHITE)
            elif val >= 0.3:
                ax.text(0, bot + val/2, f"{val:.1f}%", ha="center", va="center",
                        fontsize=8, fontweight="bold", color=WHITE)
            bot += val
        ax.set_ylim(0, 105)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

    axes[0].set_ylabel("% of Inquiries")

    # Media share callouts at bottom
    for ax, ms in zip(axes, [sem_pct+soc_pct, sem_pct2+soc_pct2, media_pct3]):
        ax.set_xlabel(f"Media: {ms:.1f}%", fontsize=11, fontweight="bold", color=CPX_RED)

    legend_items = [
        Patch(facecolor=CPX_MGREY, label="Non-Media (Baseline + Trend + Seasonality)"),
        Patch(facecolor=CPX_RED, label="G&E Social"),
        Patch(facecolor=CPX_NAVY, label="G&E SEM"),
    ]
    fig.legend(handles=legend_items, loc="lower center", ncol=3, fontsize=9)
    fig.suptitle("Media Attribution: Multiplicative Model Gives Media Fair Credit",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0.06, 1, 0.94])
    plt.show()

    print(f"\n── Summary ──")
    print(f"V1 → V3 media share: {sem_pct+soc_pct:.1f}% → {media_pct3:.1f}%")
    print(f"Baseline reduction:  {base_pct:.1f}% → {base_pct3:.1f}%  ({base_pct3-base_pct:+.1f}pp)")
    print(f"V3 media = {media_incremental:,.0f} incremental inquiries over 165 weeks")

## 11. V4 — B-Spline Baseline + Weibull Adstock

**Two structural upgrades over V3:**

| Component | V3 (current) | V4 (new) | Why |
|---|---|---|---|
| Baseline trend | Linear (forced constant slope) | **B-spline** (8 knots, random-walk prior) | Real decline is nonlinear — fast in 2024, stabilizing 2025. Linear trend either over/under-attributes to media depending on period |
| Adstock | Geometric (exponential decay from day 0) | **Weibull** (learnable peak + decay) | G&E has a consideration window: ad → browse → inquiry 2–4 wks later. Geometric can’t model a *delayed* peak — it always peaks at lag 0 |

**B-spline baseline:** 8 interior knots → ~10 basis functions. Random-walk prior on coefficients ensures smoothness (no overfitting to weekly noise). Absorbs structural trends flexibly without stealing media-correlated variance.

**Weibull adstock:** PDF shape with 2 parameters per channel:
- `shape (k)`: k=1 = exponential (same as geometric), k>1 = delayed peak (effect builds then decays)
- `scale (λ)`: controls how quickly effect dissipates

This lets SEM (immediate response) have a different lag profile than Social (awareness → consideration → inquiry pipeline).

In [0]:
# ── V4: B-Spline baseline + Weibull adstock ─────────────────────────────────
import numpy as np
import pandas as pd
from scipy.interpolate import BSpline
from scipy.stats import weibull_min

# ── 1. B-Spline basis for nonlinear trend ───────────────────────────────
n_knots = 8   # interior knots
spline_degree = 3

# Time index normalized to [0, 1]
t_norm = np.linspace(0, 1, n_weeks)

# Create knot sequence (uniform interior + boundary padding)
interior_knots = np.linspace(0, 1, n_knots + 2)[1:-1]  # exclude endpoints
knots = np.concatenate([
    np.zeros(spline_degree + 1),        # left boundary
    interior_knots,
    np.ones(spline_degree + 1)           # right boundary
])
n_bases = len(knots) - spline_degree - 1  # number of basis functions

# Evaluate B-spline basis matrix
B_spline = np.zeros((n_weeks, n_bases))
for i in range(n_bases):
    coeffs = np.zeros(n_bases)
    coeffs[i] = 1.0
    spl = BSpline(knots, coeffs, spline_degree)
    B_spline[:, i] = spl(t_norm)

print(f"B-spline basis: {n_bases} functions, degree {spline_degree}, {n_knots} interior knots")

# ── 2. Weibull adstock weights (precompute lag structure) ───────────────
MAX_LAG_V4 = 12
lag_range = np.arange(1, MAX_LAG_V4 + 1).astype(float)  # 1-indexed (lag 0 = current week)

def weibull_weights_np(shape_k, scale_lam):
    """Compute normalized Weibull PDF weights over lag_range."""
    w = weibull_min.pdf(lag_range, shape_k, scale=scale_lam)
    return w / w.sum()

# Visualize example lag profiles
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 4))
for k, lam, label in [(1.0, 3.0, "Geometric-like (k=1)"),
                       (2.0, 4.0, "Delayed peak (k=2, λ=4)"),
                       (3.0, 5.0, "Strong delay (k=3, λ=5)")]:
    w = weibull_weights_np(k, lam)
    ax.plot(lag_range, w, "o-", lw=2, label=label)
ax.set_xlabel("Lag (weeks)")
ax.set_ylabel("Weight")
ax.set_title("Weibull Adstock: Flexible Lag Profiles")
ax.legend()
plt.tight_layout()
plt.show()

# Reuse V3 data arrays
print(f"\nV4 data: {n_weeks} weeks, log(y) target")
print(f"Baseline: B-spline ({n_bases} basis fns) replaces linear trend")
print(f"Adstock: Weibull (shape+scale per channel, {MAX_LAG_V4}-week window)")
print(f"✓ V4 setup ready")

In [0]:
# ── V4 Model: B-Spline baseline + Weibull adstock + Hill + log-link ───────
import pymc as pm
import pytensor.tensor as pt
import arviz as az

def weibull_adstock_pt(x_np, shape_k, scale_lam):
    """Apply Weibull-weighted distributed lag to a spend series."""
    L = MAX_LAG_V4
    # Weibull PDF weights: (k/lam) * (t/lam)^(k-1) * exp(-(t/lam)^k)
    lags = pt.arange(1, L + 1).astype("float64")
    t_scaled = lags / scale_lam
    log_w = (pt.log(shape_k) - pt.log(scale_lam)
             + (shape_k - 1) * pt.log(t_scaled)
             - pt.power(t_scaled, shape_k))
    w = pt.exp(log_w)
    w = w / w.sum()  # normalize

    # Convolve: for each time t, weighted sum of past L spend values
    padded = pt.concatenate([pt.zeros(L), pt.as_tensor_variable(x_np)])
    idx = pt.arange(len(x_np))[:, None] + pt.arange(L)[None, :]
    # idx[t, j] = spend at time (t + L - 1 - j) => lag j+1
    return padded[idx].dot(w[::-1])

def hill_pt_v4(x, K, n):
    xn = pt.power(x + 1e-8, n)
    return xn / (pt.power(K, n) + xn)

with pm.Model() as model_v4:
    # ── B-Spline baseline (random-walk prior for smoothness) ──
    # First coeff is free, subsequent ones are increments
    spline_sigma = pm.HalfNormal("spline_sigma", 0.1)
    spline_coeff_0 = pm.Normal("spline_c0", log_y_v3.mean(), 0.3)
    spline_increments = pm.Normal("spline_diffs", 0, spline_sigma, shape=n_bases - 1)
    spline_coeffs = pt.concatenate([[spline_coeff_0],
                                     spline_coeff_0 + pt.cumsum(spline_increments)])
    log_baseline = pt.dot(pt.as_tensor_variable(B_spline), spline_coeffs)

    # ── Weibull adstock ──
    # SEM: likely more immediate (shape ~1-2)
    k_sem = pm.Gamma("k_sem", mu=1.5, sigma=0.7)   # shape
    lam_sem = pm.Gamma("lam_sem", mu=3.0, sigma=1.5)  # scale (weeks)
    # Social: likely delayed peak (shape ~2-3)
    k_soc = pm.Gamma("k_soc", mu=2.5, sigma=1.0)
    lam_soc = pm.Gamma("lam_soc", mu=4.0, sigma=1.5)

    sem_ad = weibull_adstock_pt(sem_n_v3, k_sem, lam_sem)
    soc_ad = weibull_adstock_pt(soc_n_v3, k_soc, lam_soc)

    # ── Hill saturation ──
    K_sem = pm.Beta("K_sem", 2, 2)
    K_soc = pm.Beta("K_soc", 2, 2)
    n_sem = pm.Gamma("n_sem", 2, 2)
    n_soc = pm.Gamma("n_soc", 2, 2)
    sem_eff = hill_pt_v4(sem_ad, K_sem, n_sem)
    soc_eff = hill_pt_v4(soc_ad, K_soc, n_soc)

    # ── Media betas (log-space % lifts) ──
    b_sem = pm.HalfNormal("beta_sem", 0.25)
    b_soc = pm.HalfNormal("beta_soc", 0.25)

    # ── Seasonality (same G&E features) ──
    b_seas = pm.Normal("beta_seas", 0, 0.1, shape=Xs_v3.shape[1])

    # ── Tracked contributions ──
    media_sem = pm.Deterministic("media_sem", b_sem * sem_eff)
    media_soc = pm.Deterministic("media_soc", b_soc * soc_eff)

    # ── Full model: log(y) = spline_baseline + media + seasonality ──
    log_mu = log_baseline + media_sem + media_soc + pt.dot(Xs_v3, b_seas)

    sigma = pm.HalfNormal("sigma", 0.12)
    pm.Normal("log_y", mu=log_mu, sigma=sigma, observed=log_y_v3)

    idata_v4 = pm.sample(1500, tune=2500, chains=4, target_accept=0.995,
                         nuts_sampler="numpyro", random_seed=42)

print("✓ V4 model sampling complete.")

In [0]:
# ── V4 Results — Counterfactual Decomposition + MAPE ─────────────────────
print("="*70)
print("  V4 RESULTS: B-Spline Baseline + Weibull Adstock")
print("="*70)

# Key posteriors
s_v4 = az.summary(idata_v4, var_names=["beta_sem", "beta_soc",
                                        "k_sem", "lam_sem", "k_soc", "lam_soc",
                                        "K_sem", "K_soc", "n_sem", "n_soc",
                                        "spline_sigma", "sigma"])
print("\n── V4 Posteriors ──")
print(s_v4.to_string())

div_v4 = int(idata_v4.sample_stats.diverging.sum())
print(f"\nDivergences: {div_v4}")

# ── Counterfactual Decomposition ──
post_v4 = idata_v4.posterior
media_sem_v4 = post_v4["media_sem"].mean(dim=["chain", "draw"]).values
media_soc_v4 = post_v4["media_soc"].mean(dim=["chain", "draw"]).values

# Reconstruct spline baseline
spline_c0_mean = float(post_v4["spline_c0"].mean())
spline_diffs_mean = post_v4["spline_diffs"].mean(dim=["chain", "draw"]).values
spline_coeffs_mean = np.concatenate([[spline_c0_mean],
                                      spline_c0_mean + np.cumsum(spline_diffs_mean)])
log_baseline_mean = B_spline @ spline_coeffs_mean

b_seas_v4_mean = post_v4["beta_seas"].mean(dim=["chain", "draw"]).values

# Full prediction
log_pred_full_v4 = log_baseline_mean + media_sem_v4 + media_soc_v4 + Xs_v3 @ b_seas_v4_mean
pred_full_v4 = np.exp(log_pred_full_v4)

# Counterfactual (no media)
log_pred_no_media_v4 = log_baseline_mean + Xs_v3 @ b_seas_v4_mean
pred_no_media_v4 = np.exp(log_pred_no_media_v4)

# Incremental
media_incr_v4 = pred_full_v4.sum() - pred_no_media_v4.sum()
sem_incr_v4 = (np.exp(log_pred_full_v4) - np.exp(log_pred_full_v4 - media_sem_v4)).sum()
soc_incr_v4 = (np.exp(log_pred_full_v4) - np.exp(log_pred_full_v4 - media_soc_v4)).sum()

sem_pct4 = sem_incr_v4 / actual_total * 100
soc_pct4 = soc_incr_v4 / actual_total * 100
media_pct4 = media_incr_v4 / actual_total * 100
base_pct4 = 100 - media_pct4

print("\n── Counterfactual Decomposition ──")
print(f"  G&E SEM:    {sem_pct4:.1f}%  ({sem_incr_v4:,.0f} incremental inquiries)")
print(f"  G&E Social: {soc_pct4:.1f}%  ({soc_incr_v4:,.0f} incremental inquiries)")
print(f"  Total media: {media_pct4:.1f}%  ({media_incr_v4:,.0f} incremental inquiries)")
print(f"  Non-media:   {base_pct4:.1f}%")

# ── MAPE ──
split_idx = int(n_weeks * 0.8)
mape_train_v4 = np.mean(np.abs((y_v3[:split_idx] - pred_full_v4[:split_idx]) / y_v3[:split_idx])) * 100
mape_val_v4 = np.mean(np.abs((y_v3[split_idx:] - pred_full_v4[split_idx:]) / y_v3[split_idx:])) * 100

print("\n── MAPE ──")
print(f"  V4 Train: {mape_train_v4:.1f}%   Val: {mape_val_v4:.1f}%")
print(f"  V3 Train: 16.6%   Val: 17.4%  (for reference)")

# ── Full comparison ──
print("\n── Media Share Across All Versions ──")
print(f"  V1:  {sem_pct + soc_pct:.1f}%  (additive/tanh/geometric/linear)")
print(f"  V2:  {sem_pct2 + soc_pct2:.1f}%  (additive/tanh/geometric/G&E seas)")
print(f"  V3:  {media_pct3:.1f}%  (multiplicative/Hill/geometric/linear trend)")
print(f"  V4: {media_pct4:.1f}%  (multiplicative/Hill/Weibull/B-spline)")

# ── Weibull lag interpretation ──
k_sem_mean = float(s_v4.loc["k_sem", "mean"])
lam_sem_mean = float(s_v4.loc["lam_sem", "mean"])
k_soc_mean = float(s_v4.loc["k_soc", "mean"])
lam_soc_mean = float(s_v4.loc["lam_soc", "mean"])

# Peak lag for Weibull: mode = lam * ((k-1)/k)^(1/k) for k>1
if k_sem_mean > 1:
    peak_sem = lam_sem_mean * ((k_sem_mean - 1) / k_sem_mean) ** (1 / k_sem_mean)
else:
    peak_sem = 0  # exponential decay, peak at lag 0
if k_soc_mean > 1:
    peak_soc = lam_soc_mean * ((k_soc_mean - 1) / k_soc_mean) ** (1 / k_soc_mean)
else:
    peak_soc = 0

print(f"\n── Weibull Lag Profiles ──")
print(f"  SEM: shape={k_sem_mean:.2f}, scale={lam_sem_mean:.1f}wk → peak effect at week {peak_sem:.1f}")
print(f"  Soc: shape={k_soc_mean:.2f}, scale={lam_soc_mean:.1f}wk → peak effect at week {peak_soc:.1f}")

In [0]:
# ── V4 Diagnostic Plots ─────────────────────────────────────────────────
from scipy.stats import weibull_min
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 2, figsize=get_figsize("16:9"))

# ── Panel A: Spline baseline vs linear trend ──
ax = axes[0, 0]
week_dates_plot = df_v3.index
ax.plot(week_dates_plot, np.exp(log_baseline_mean), color=CPX_NAVY, lw=2, label="V4: B-Spline baseline")
ax.plot(week_dates_plot, np.exp(log_base_mean + float(s_v3.loc['beta_trend','mean']) * trend_v3),
        color=CPX_MGREY, lw=2, ls="--", label="V3: Linear trend")
ax.plot(week_dates_plot, y_v3, color=CPX_RED, alpha=0.3, lw=0.8, label="Actual inquiries")
ax.set_ylabel("Inquiries/wk")
ax.set_title("Spline Captures Nonlinear Decline")
ax.legend(fontsize=8)

# ── Panel B: Learned Weibull lag profiles ──
ax = axes[0, 1]
lags_plot = np.arange(1, MAX_LAG_V4 + 1)
w_sem = weibull_min.pdf(lags_plot, k_sem_mean, scale=lam_sem_mean)
w_sem = w_sem / w_sem.sum()
w_soc = weibull_min.pdf(lags_plot, k_soc_mean, scale=lam_soc_mean)
w_soc = w_soc / w_soc.sum()

ax.bar(lags_plot - 0.18, w_sem, width=0.35, color=CPX_NAVY, label=f"SEM (k={k_sem_mean:.1f}, λ={lam_sem_mean:.1f})")
ax.bar(lags_plot + 0.18, w_soc, width=0.35, color=CPX_RED, label=f"Social (k={k_soc_mean:.1f}, λ={lam_soc_mean:.1f})")
ax.set_xlabel("Lag (weeks)")
ax.set_ylabel("Weight")
ax.set_title("Learned Lag Profiles (Weibull)")
ax.legend(fontsize=8)

# ── Panel C: Predicted vs Actual ──
ax = axes[1, 0]
ax.plot(week_dates_plot, y_v3, color=CPX_DGREY, lw=1, alpha=0.7, label="Actual")
ax.plot(week_dates_plot, pred_full_v4, color=CPX_RED, lw=1.5, label="V4 predicted")
ax.axvline(week_dates_plot[split_idx], ls="--", color=CPX_GOLD, lw=1, label="Train/Val split")
ax.set_ylabel("Inquiries/wk")
ax.set_title(f"Fit: Train MAPE={mape_train_v4:.1f}%, Val MAPE={mape_val_v4:.1f}%")
ax.legend(fontsize=8)

# ── Panel D: Decomposition comparison ──
ax = axes[1, 1]
models = ["V1", "V2", "V3", "V4"]
media_shares = [sem_pct+soc_pct, sem_pct2+soc_pct2, media_pct3, media_pct4]
base_shares = [base_pct, base_pct2, base_pct3, base_pct4]

x = np.arange(4)
ax.bar(x, base_shares, color=CPX_MGREY, label="Non-Media")
ax.bar(x, media_shares, bottom=base_shares, color=CPX_RED, label="Media")
for i, ms in enumerate(media_shares):
    ax.text(i, base_shares[i] + ms/2, f"{ms:.1f}%", ha="center", va="center",
            fontsize=10, fontweight="bold", color=WHITE)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.set_title("Media Share Progression")
ax.legend(fontsize=8)

fig.suptitle("V4: Spline Baseline + Weibull Adstock — Diagnostic Panel",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print(f"\n── Final Summary ──")
print(f"  V1 → V4 media share: {sem_pct+soc_pct:.1f}% → {media_pct4:.1f}%")
print(f"  Baseline: {base_pct:.1f}% → {base_pct4:.1f}%  ({base_pct4-base_pct:+.1f}pp)")
print(f"  Incremental inquiries from media: {media_incr_v4:,.0f} over 165 weeks")
print(f"  MAPE improvement: V1 val 39.9% → V4 val {mape_val_v4:.1f}%")

## 12. V5 — Fixing SEM Undercrediting

**Problem:** V4 gives SEM only 4.2% vs Social 5.8%, despite SEM being a direct-response channel.

**Root cause:** The Weibull adstock starts at **lag 1** (next week). For SEM (someone Googles “private screening Toronto” → clicks → fills inquiry form *same day*), the strongest signal is at **lag 0** (same week). We’re discarding SEM’s highest-intent conversions.

**Fix — Spike-and-Carry adstock for SEM:**
```
SEM_effect = α × spend_this_week + (1-α) × Weibull_carry(past_weeks)
```
- `α` (learnable, Beta prior): fraction of SEM effect that’s immediate
- `(1-α)`: residual carry-over for people who searched, browsed, came back later
- Social stays pure Weibull (no lag-0 component) — awareness channels don’t convert same-week

This is structurally correct: SEM is high-intent, Social is awareness.

In [0]:
# ── V5: SEM gets spike-and-carry adstock (lag 0 + Weibull) ───────────────
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import numpy as np

def weibull_adstock_v5(x_np, shape_k, scale_lam):
    """Weibull-weighted lag starting at lag 1 (no same-week)."""
    L = MAX_LAG_V4
    lags = pt.arange(1, L + 1).astype("float64")
    t_scaled = lags / scale_lam
    log_w = (pt.log(shape_k) - pt.log(scale_lam)
             + (shape_k - 1) * pt.log(t_scaled)
             - pt.power(t_scaled, shape_k))
    w = pt.exp(log_w)
    w = w / w.sum()
    padded = pt.concatenate([pt.zeros(L), pt.as_tensor_variable(x_np)])
    idx = pt.arange(len(x_np))[:, None] + pt.arange(L)[None, :]
    return padded[idx].dot(w[::-1])

def hill_v5(x, K, n):
    xn = pt.power(x + 1e-8, n)
    return xn / (pt.power(K, n) + xn)

with pm.Model() as model_v5:
    # ── B-Spline baseline (same as V4) ──
    spline_sigma = pm.HalfNormal("spline_sigma", 0.1)
    spline_coeff_0 = pm.Normal("spline_c0", log_y_v3.mean(), 0.3)
    spline_increments = pm.Normal("spline_diffs", 0, spline_sigma, shape=n_bases - 1)
    spline_coeffs = pt.concatenate([[spline_coeff_0],
                                     spline_coeff_0 + pt.cumsum(spline_increments)])
    log_baseline = pt.dot(pt.as_tensor_variable(B_spline), spline_coeffs)

    # ── SEM: Spike-and-Carry Adstock ──
    # α = fraction of immediate (same-week) response
    alpha_sem = pm.Beta("alpha_sem", 4, 2)  # prior: ~60-70% immediate (high-intent channel)
    k_sem = pm.Gamma("k_sem", mu=1.5, sigma=0.7)
    lam_sem = pm.Gamma("lam_sem", mu=3.0, sigma=1.5)
    # Spike: current-week spend (normalized)
    sem_immediate = pt.as_tensor_variable(sem_n_v3)
    # Carry: Weibull-weighted past weeks
    sem_carry = weibull_adstock_v5(sem_n_v3, k_sem, lam_sem)
    # Combined
    sem_ad = alpha_sem * sem_immediate + (1 - alpha_sem) * sem_carry

    # ── Social: Pure Weibull (awareness, no same-week response) ──
    k_soc = pm.Gamma("k_soc", mu=2.5, sigma=1.0)
    lam_soc = pm.Gamma("lam_soc", mu=4.0, sigma=1.5)
    soc_ad = weibull_adstock_v5(soc_n_v3, k_soc, lam_soc)

    # ── Hill saturation ──
    K_sem = pm.Beta("K_sem", 2, 3)    # SEM: lower K = saturates later (more headroom)
    K_soc = pm.Beta("K_soc", 2, 2)
    n_sem = pm.Gamma("n_sem", 2, 2)
    n_soc = pm.Gamma("n_soc", 2, 2)
    sem_eff = hill_v5(sem_ad, K_sem, n_sem)
    soc_eff = hill_v5(soc_ad, K_soc, n_soc)

    # ── Media betas (log-space) ──
    b_sem = pm.HalfNormal("beta_sem", 0.3)  # slightly wider prior for SEM
    b_soc = pm.HalfNormal("beta_soc", 0.25)

    # ── Seasonality ──
    b_seas = pm.Normal("beta_seas", 0, 0.1, shape=Xs_v3.shape[1])

    # ── Tracked contributions ──
    media_sem = pm.Deterministic("media_sem", b_sem * sem_eff)
    media_soc = pm.Deterministic("media_soc", b_soc * soc_eff)

    # ── Full model ──
    log_mu = log_baseline + media_sem + media_soc + pt.dot(Xs_v3, b_seas)
    sigma = pm.HalfNormal("sigma", 0.12)
    pm.Normal("log_y", mu=log_mu, sigma=sigma, observed=log_y_v3)

    idata_v5 = pm.sample(1500, tune=2500, chains=4, target_accept=0.995,
                         nuts_sampler="numpyro", random_seed=42)

print("✓ V5 model sampling complete.")

In [0]:
# ── V5 Results & Comparison ─────────────────────────────────────────────
print("="*70)
print("  V5 RESULTS: SEM Spike-and-Carry + Social Weibull")
print("="*70)

s_v5 = az.summary(idata_v5, var_names=["beta_sem", "beta_soc", "alpha_sem",
                                        "k_sem", "lam_sem", "k_soc", "lam_soc",
                                        "K_sem", "K_soc", "spline_sigma", "sigma"])
print("\n── V5 Posteriors ──")
print(s_v5.to_string())

div_v5 = int(idata_v5.sample_stats.diverging.sum())
print(f"\nDivergences: {div_v5}")

# ── Counterfactual Decomposition ──
post_v5 = idata_v5.posterior
media_sem_v5 = post_v5["media_sem"].mean(dim=["chain", "draw"]).values
media_soc_v5 = post_v5["media_soc"].mean(dim=["chain", "draw"]).values

# Reconstruct spline baseline
c0_v5 = float(post_v5["spline_c0"].mean())
diffs_v5 = post_v5["spline_diffs"].mean(dim=["chain", "draw"]).values
coeffs_v5 = np.concatenate([[c0_v5], c0_v5 + np.cumsum(diffs_v5)])
log_bl_v5 = B_spline @ coeffs_v5
b_seas_v5 = post_v5["beta_seas"].mean(dim=["chain", "draw"]).values

# Full prediction
log_full_v5 = log_bl_v5 + media_sem_v5 + media_soc_v5 + Xs_v3 @ b_seas_v5
pred_full_v5 = np.exp(log_full_v5)

# Counterfactual (no media)
log_no_media_v5 = log_bl_v5 + Xs_v3 @ b_seas_v5
pred_no_media_v5 = np.exp(log_no_media_v5)

# Incremental
sem_incr_v5 = (np.exp(log_full_v5) - np.exp(log_full_v5 - media_sem_v5)).sum()
soc_incr_v5 = (np.exp(log_full_v5) - np.exp(log_full_v5 - media_soc_v5)).sum()
media_incr_v5 = pred_full_v5.sum() - pred_no_media_v5.sum()

sem_pct5 = sem_incr_v5 / actual_total * 100
soc_pct5 = soc_incr_v5 / actual_total * 100
media_pct5 = media_incr_v5 / actual_total * 100
base_pct5 = 100 - media_pct5

# MAPE
split_idx = int(len(y_v3) * 0.8)
actual_total = y_v3.sum()
mape_train_v5 = np.mean(np.abs((y_v3[:split_idx] - pred_full_v5[:split_idx]) / y_v3[:split_idx])) * 100
mape_val_v5 = np.mean(np.abs((y_v3[split_idx:] - pred_full_v5[split_idx:]) / y_v3[split_idx:])) * 100

print("\n── Counterfactual Decomposition ──")
print(f"  G&E SEM:    {sem_pct5:.1f}%  ({sem_incr_v5:,.0f} incremental inquiries)")
print(f"  G&E Social: {soc_pct5:.1f}%  ({soc_incr_v5:,.0f} incremental inquiries)")
print(f"  Total media: {media_pct5:.1f}%  ({media_incr_v5:,.0f} inquiries)")
print(f"  Non-media:   {base_pct5:.1f}%")

print("\n── SEM Rebalancing ──")
alpha_mean = float(s_v5.loc["alpha_sem", "mean"])
print(f"  alpha_sem = {alpha_mean:.2f} → {alpha_mean*100:.0f}% of SEM effect is SAME-WEEK")
print(f"  V4 SEM: 4.2%  →  V5 SEM: {sem_pct5:.1f}%  (Δ={sem_pct5-4.2:+.1f}pp)")
print(f"  V4 Soc: 5.8%  →  V5 Soc: {soc_pct5:.1f}%  (Δ={soc_pct5-5.8:+.1f}pp)")

print("\n── MAPE ──")
print(f"  V5: Train {mape_train_v5:.1f}%  Val {mape_val_v5:.1f}%")
print(f"  V4: Train 16.1%  Val 16.1%")

print("\n── Full Progression ──")
print(f"{'Version':<12} {'Media':>7} {'SEM':>7} {'Social':>7} {'Non-Media':>10} {'Val MAPE':>9}")
print("-"*55)
print(f"{'V1':<12} {'3.3%':>7} {'0.6%':>7} {'2.7%':>7} {'96.7%':>10} {'39.9%':>9}")
print(f"{'V2':<12} {'4.9%':>7} {'0.5%':>7} {'4.4%':>7} {'95.1%':>10} {'33.0%':>9}")
print(f"{'V3':<12} {'9.8%':>7} {'3.2%':>7} {'6.8%':>7} {'90.2%':>10} {'17.4%':>9}")
print(f"{'V4':<12} {'9.8%':>7} {'4.2%':>7} {'5.8%':>7} {'90.2%':>10} {'16.1%':>9}")
print(f"{'V5':<12} {media_pct5:>6.1f}% {sem_pct5:>6.1f}% {soc_pct5:>6.1f}% {base_pct5:>9.1f}% {mape_val_v5:>8.1f}%")

In [0]:
# ── V5 Decay Profiles: SEM (Spike+Carry) vs Social (Pure Weibull) ────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import weibull_min

# V5 posteriors
alpha_mean = float(s_v5.loc["alpha_sem", "mean"])
k_sem_mean = float(s_v5.loc["k_sem", "mean"])
lam_sem_mean = float(s_v5.loc["lam_sem", "mean"])
k_soc_mean = float(s_v5.loc["k_soc", "mean"])
lam_soc_mean = float(s_v5.loc["lam_soc", "mean"])

# Lag range including lag 0
lags_full = np.arange(0, MAX_LAG_V4 + 1).astype(float)
lags_carry = np.arange(1, MAX_LAG_V4 + 1).astype(float)

# ── SEM: Spike-and-Carry profile ──
w_carry = weibull_min.pdf(lags_carry, k_sem_mean, scale=lam_sem_mean)
w_carry = w_carry / w_carry.sum()
sem_profile = np.zeros(len(lags_full))
sem_profile[0] = alpha_mean           # spike at lag 0 (same week)
sem_profile[1:] = (1 - alpha_mean) * w_carry  # carry at lags 1-12

# ── Social: Pure Weibull (no lag 0) ──
soc_profile = np.zeros(len(lags_full))
w_soc = weibull_min.pdf(lags_carry, k_soc_mean, scale=lam_soc_mean)
w_soc = w_soc / w_soc.sum()
soc_profile[1:] = w_soc

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=get_figsize("16:9"), sharey=True)

# Left: SEM Spike-and-Carry
bars = ax1.bar(lags_full, sem_profile, color=CPX_NAVY, width=0.7, edgecolor=WHITE, linewidth=0.5)
bars[0].set_color(CPX_RED)  # highlight the spike
ax1.set_xlabel("Lag (weeks)")
ax1.set_ylabel("Weight (normalized)")
ax1.set_title("SEM: Spike-and-Carry Adstock")
ax1.set_xticks(lags_full)
ax1.set_xticklabels([f"{int(l)}" for l in lags_full])

# Annotate spike
ax1.annotate(f"Spike: α={alpha_mean:.0%}\n(same-week)", 
             xy=(0, sem_profile[0]), xytext=(3, sem_profile[0]*0.85),
             fontsize=9, fontweight="bold", color=CPX_RED,
             arrowprops=dict(arrowstyle="->", color=CPX_RED, lw=1.5))
ax1.annotate(f"Carry: {1-alpha_mean:.0%}\n(Weibull k={k_sem_mean:.1f}, λ={lam_sem_mean:.1f})", 
             xy=(3, sem_profile[3]), xytext=(6, sem_profile[0]*0.45),
             fontsize=9, fontweight="bold", color=CPX_NAVY,
             arrowprops=dict(arrowstyle="->", color=CPX_NAVY, lw=1.5))

# Right: Social Pure Weibull
ax2.bar(lags_full, soc_profile, color=CPX_RED, width=0.7, edgecolor=WHITE, linewidth=0.5)
ax2.set_xlabel("Lag (weeks)")
ax2.set_title("Social: Pure Weibull (No Same-Week)")
ax2.set_xticks(lags_full)
ax2.set_xticklabels([f"{int(l)}" for l in lags_full])

# Annotate peak
peak_soc = lam_soc_mean * ((k_soc_mean - 1) / k_soc_mean) ** (1 / k_soc_mean)
ax2.annotate(f"Peak at week {peak_soc:.1f}\n(k={k_soc_mean:.1f}, λ={lam_soc_mean:.1f})", 
             xy=(round(peak_soc), soc_profile[round(peak_soc)]),
             xytext=(7, soc_profile[round(peak_soc)]*1.5),
             fontsize=9, fontweight="bold", color=CPX_RED,
             arrowprops=dict(arrowstyle="->", color=CPX_RED, lw=1.5))

# No spike label
ax2.text(0, 0.005, "No lag-0\n(awareness)", ha="center", va="bottom",
         fontsize=8, color=CPX_MGREY, fontstyle="italic")

fig.suptitle("V5 Learned Decay Profiles — SEM (Immediate + Carry) vs Social (Delayed Peak)",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"\nSEM: {alpha_mean:.0%} immediate (lag 0) + {1-alpha_mean:.0%} Weibull carry (k={k_sem_mean:.1f}, λ={lam_sem_mean:.1f})")
print(f"Social: 0% immediate + 100% Weibull (peak at week {peak_soc:.1f})")
print(f"\nV1–V4 had no lag-0 bar for SEM — discarding 73% of its signal into baseline.")